In [25]:
import pandas as pd
import numpy as np

In [26]:
# checking encoding type
meta = pd.read_csv('data.processed/meta_extract.csv', encoding='utf-8') 
meta.head(20)

,url,title,runtime,rating,metascore,userscore,genre,cast,director,studio,awards
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,83.0,| Not Rated,70,NaN,Documentary,NaN,Lynn Hershman-Leeson,Hotwire Productions,NaN
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,104.0,| PG-13,76,7.7,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,Paramount Pictures,"#18MostDiscussedMovieof2016 , #1MostSharedMovi..."
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,82.0,| R,54,5.8,"Drama,Comedy,Romance","Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,Click Star,NaN
3,https://www.metacritic.com/movie/10-years,10 Years,100.0,| R,61,6.9,"Drama,Comedy,Romance","Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,Anchor Bay Entertainment,NaN
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,91.0,| Not Rated,63,7.5,"Horror,Comedy",NaN,Cameron Cairnes,Music Box Films,NaN
5,https://www.metacritic.com/movie/100-streets,100 Streets,93.0,NaN,44,6.1,Drama,NaN,Jim O'Hanlon,Samuel Goldwyn Films,NaN
6,https://www.metacritic.com/movie/1000-times-go...,"1,000 Times Good Night",117.0,| Not Rated,57,6.8,Drama,NaN,Erik Poppe,Film Movement,NaN
7,https://www.metacritic.com/movie/10000-bc,"10,000 BC",109.0,| PG-13,34,4.6,"Adventure,Drama,Fantasy","Camilla Belle,Marco Khan,Steven Strait",Roland Emmerich,Warner Bros. Pictures,"#23MostDiscussedMovieof2008 , #27MostSharedMov..."
8,https://www.metacritic.com/movie/10000-km,"10,000 km",99.0,| R,75,7.4,"Drama,Comedy,Romance",NaN,Carlos Marques-Marcet,Broad Green Pictures,NaN
9,https://www.metacritic.com/movie/1001-grams,1001 Grams,93.0,| Not Rated,65,NaN,Drama,NaN,Bent Hamer,Kino Lorber,NaN


In [27]:
# detect the real encoding
import chardet

with open('data.processed/meta_extract.csv', 'rb') as f:
    result = chardet.detect(f.read(100000))
print(result)

{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}


In [28]:
# test encoding on one string
test = "GÃ©rard Depardieu"
fixed = test.encode('latin1').decode('utf-8')
print(fixed)

Gérard Depardieu


In [29]:
# apply encoding to the real column
meta['cast'] = meta['cast'].str.encode('latin1', errors='ignore').str.decode('utf-8', errors='ignore')
meta['cast'].head(20)

0                                                   NaN
1     John Gallagher Jr.,John Goodman,Mary Elizabeth...
2                    Jonah Hill,Morgan Freeman,Paz Vega
3                Channing Tatum,Chris Pratt,Jenna Dewan
4                                                   NaN
5                                                   NaN
6                                                   NaN
7                Camilla Belle,Marco Khan,Steven Strait
8                                                   NaN
9                                                   NaN
10                  Hilmir Snær Guðnason,Victoria Abril
11                         Gérard Depardieu,Glenn Close
12            Brad Renfro,Giovanni Ribisi,James Marsden
13                                                  NaN
14                                                  NaN
15                                           Abby Linne
16       Apti Magamaev,Sergei Makovetsky,Sergey Garmash
17                                              

In [30]:
# split the cast column into individual actors
cast_long = meta['cast'].dropna().str.split(',').explode().str.strip()

In [31]:
# build the unique cast_member table
cast_member = (
    cast_long
    .drop_duplicates()
    .to_frame(name='cast_name')
    .reset_index(drop=True)
)
cast_member.insert(0, 'cast_id', cast_member.index + 1)
cast_member.head(20)

,cast_id,cast_name
0,1,John Gallagher Jr.
1,2,John Goodman
2,3,Mary Elizabeth Winstead
3,4,Jonah Hill
4,5,Morgan Freeman
5,6,Paz Vega
6,7,Channing Tatum
7,8,Chris Pratt
8,9,Jenna Dewan
9,10,Camilla Belle


In [32]:
cast_member.info

<bound method DataFrame.info of        cast_id                cast_name
0            1       John Gallagher Jr.
1            2             John Goodman
2            3  Mary Elizabeth Winstead
3            4               Jonah Hill
4            5           Morgan Freeman
...        ...                      ...
24857    24858           Ninon François
24858    24859        Patrick Boucheron
24859    24860          Sayyid El Alami
24860    24861         Wislanda Louimat
24861    24862            Victoria Hall

[24862 rows x 2 columns]>

In [ ]:
# export the result
cast_member.to_csv('data.processed/cast_member_extract.csv', index=False, encoding='utf-8')

In [35]:
# verify by reopening the exported file
check = pd.read_csv('data.processed/cast_member_extract.csv')
check[check['cast_name'].str.contains('nnaeson|rard', na=False)]

,cast_id,cast_name
14,15,Gérard Depardieu
344,345,Gerard Butler
3267,3268,Ana Girardot
3704,3705,Gerard Horan
4806,4807,Nolan Gerard Funk
4997,4998,Simone-Élise Girard
5956,5957,Luis Gerardo M
6549,6550,Gerard Catus
7500,7501,Eliza Brady-Girard
7684,7685,Ella-June Henrard


In [36]:
cast_member.tail(20)

,cast_id,cast_name
24842,24843,Dennis Mallen
24843,24844,Megan Hayes
24844,24845,Nasir Rahim
24845,24846,Nelcie Souffrant
24846,24847,Pedro Tavarez Jr.
24847,24848,Steve Heinz
24848,24849,Tony Demil
24849,24850,Ts Madison
24850,24851,Adilé David
24851,24852,Ginite Popote


In [37]:
cast_member.shape

(24862, 2)